# 🚀 SDXL LoRA 训练环境测试 - 车柱完角色

这个 Notebook 会自动测试 Colab GPU 环境，并准备 LoRA 训练。

**测试步骤：**
1. 检查 GPU 配置
2. 安装 kohya_ss 训练工具
3. 下载 SDXL 1.0 模型
4. 测试训练环境
5. 准备训练配置

---

**使用前：**
- 点击右上角 **"连接到托管运行时"**
- **运行时 → 更改运行时类型 → 硬件加速器: GPU**
- 建议使用 **T4 GPU**（免费）或 **A100**（Pro）

In [ ]:
# Step 1: 检查 GPU 配置
import torch
import subprocess

print("=" * 60)
print("📊 GPU 配置检查")
print("=" * 60)

# 检查 CUDA 是否可用
print(f"\n✅ CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ GPU 数量: {gpu_count}")
    
    for i in range(gpu_count):
        gpu_name = torch.cuda.get_device_name(i)
        gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"\n🎮 GPU {i}: {gpu_name}")
        print(f"   显存: {gpu_memory:.2f} GB")
        
        # 检查是否是 T4 或 A100
        if "T4" in gpu_name:
            print("   ✅ T4 GPU 检测成功！（免费版）")
        elif "A100" in gpu_name:
            print("   ✅ A100 GPU 检测成功！（Pro 版）")
        elif "V100" in gpu_name:
            print("   ✅ V100 GPU 检测成功！")
        else:
            print(f"   ⚠️ 未知 GPU 型号: {gpu_name}")
            print("   建议：更改为 T4 或 A100 以获得最佳性能")
    
    # 测试 GPU 计算
    print(f"\n🧪 测试 GPU 计算...")
    x = torch.randn(1000, 1000).cuda()
    y = torch.matmul(x, x)
    print(f"   ✅ GPU 计算测试通过！")
    
else:
    print("\n❌ 错误: 未检测到 GPU！")
    print("   请点击：运行时 → 更改运行时类型 → 硬件加速器: GPU")
    raise RuntimeError("No GPU detected")

print("\n" + "=" * 60)
print("✅ GPU 检查完成！")
print("=" * 60)

In [ ]:
# Step 2: 安装 kohya_ss（LoRA 训练工具）
import os
from pathlib import Path

print("=" * 60)
print("📦 安装 kohya_ss 训练工具")
print("=" * 60)

# 克隆 kohya_ss 仓库
if not Path("/content/kohya_ss").exists():
    print("\n📥 克隆 kohya_ss 仓库...")
    !git clone https://github.com/bmaltais/kohya_ss /content/kohya_ss
    print("   ✅ 克隆完成！")
else:
    print("\n✅ kohya_ss 已存在，更新...")
    %cd /content/kohya_ss
    !git pull
    %cd /content
    print("   ✅ 更新完成！")

# 安装依赖
print("\n📦 安装 Python 依赖...")
%cd /content/kohya_ss
!pip install -q -r requirements.txt
!pip install -q -r requirements_linux.txt
print("   ✅ 依赖安装完成！")

# 安装 xformers（加速训练）
print("\n⚡ 安装 xformers（训练加速）...")
!pip install -q xformers
print("   ✅ xformers 安装完成！")

# 检查安装
print("\n🔍 检查安装...")
if Path("/content/kohya_ss/train_network.py").exists():
    print("   ✅ train_network.py 存在")
else:
    print("   ❌ train_network.py 不存在！")

%cd /content

print("\n" + "=" * 60)
print("✅ kohya_ss 安装完成！")
print("=" * 60)

In [ ]:
# Step 3: 下载 SDXL 1.0 模型
import os
from pathlib import Path

print("=" * 60)
print("📥 下载 SDXL 1.0 模型")
print("=" * 60)

# 创建模型目录
model_dir = Path("/content/kohya_ss/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "sd_xl_base_1.0.safetensors"

if model_path.exists():
    file_size = model_path.stat().st_size / 1024**3
    print(f"\n✅ 模型已存在: {file_size:.2f} GB")
else:
    print("\n📥 下载 SDXL 1.0 模型（~6.5GB，需要 5-10 分钟）...")
    !wget -O {model_path} \
        https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
    
    if model_path.exists():
        file_size = model_path.stat().st_size / 1024**3
        print(f"   ✅ 下载完成: {file_size:.2f} GB")
    else:
        print("   ❌ 下载失败！")
        raise RuntimeError("Model download failed")

print("\n" + "=" * 60)
print("✅ SDXL 模型准备完成！")
print("=" * 60)

In [ ]:
# Step 4: 创建训练数据目录结构
from pathlib import Path
import os

print("=" * 60)
print("📁 创建训练数据目录")
print("=" * 60)

# 创建训练数据目录（100 = repeat count）
train_data_dir = Path("/content/kohya_ss/train_data/100_chezhuhuan")
train_data_dir.mkdir(parents=True, exist_ok=True)

print(f"\n✅ 训练数据目录: {train_data_dir}")
print(f"   目录结构: train_data/100_chezhuhuan/*.jpg")
print(f"              train_data/100_chezhuhuan/*.txt")

# 创建输出目录
output_dir = Path("/content/kohya_ss/output")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n✅ 输出目录: {output_dir}")

# 创建配置文件目录
config_dir = Path("/content/kohya_ss/config")
config_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ 配置目录: {config_dir}")

print("\n" + "=" * 60)
print("✅ 目录结构创建完成！")
print("=" * 60)
print("\n📌 下一步：上传训练数据")
print("   1. 运行下一个单元格上传数据")
print("   2. 或挂载 Google Drive 复制数据")

In [ ]:
# Step 5: 准备训练数据（自动搜索 Google Drive）
from google.colab import drive
from pathlib import Path
import shutil
import os

print('=' * 60)
print('📂 准备训练数据')
print('=' * 60)

train_data_dir = Path('/content/kohya_ss/train_data/100_chezhuhuan')
train_data_dir.mkdir(parents=True, exist_ok=True)

# 挂载 Google Drive
print('
📂 挂载 Google Drive...')
drive.mount('/content/drive', force_remount=False)
print('   ✅ Google Drive 挂载完成！')

# ==========================================
# 自动搜索训练数据目录
# 支持多种可能的上传路径
# ==========================================
drive_root = Path('/content/drive/MyDrive')

# 候选路径列表（按优先级）
candidate_paths = [
    drive_root / 'lora_training_data',
    drive_root / 'lora_training_data' / '100_chezhuhuan',
    drive_root / 'training_data',
    drive_root / 'train_data',
    drive_root / '100_chezhuhuan',
    drive_root / 'chezhuhuan',
    drive_root / 'chezhuhuan_lora',
]

print('
🔍 搜索 Google Drive 中的训练数据...')
print('Drive 根目录内容:')
try:
    for item in sorted(drive_root.iterdir()):
        marker = '📁' if item.is_dir() else '📄'
        print(f'  {marker} {item.name}')
except Exception as e:
    print(f'  ❌ 无法列出目录: {e}')

# 找到实际的源目录
source_dir = None

# 先检查候选路径
for path in candidate_paths:
    if path.exists():
        # 检查是否包含图片
        imgs = list(path.glob('*.jpg')) + list(path.glob('*.png')) + list(path.glob('*.jpeg'))
        if imgs:
            source_dir = path
            print(f'
✅ 找到训练数据: {path}')
            print(f'   包含 {len(imgs)} 张图片')
            break
        else:
            print(f'  ℹ️ {path.name} 存在但不含图片')

# 如果候选路径都没找到，递归搜索
if source_dir is None:
    print('
🔍 递归搜索包含 .jpg 文件的目录...')
    for img_file in list(drive_root.rglob('*.jpg'))[:5] + list(drive_root.rglob('*.png'))[:5]:
        print(f'  找到图片: {img_file}')
        potential_dir = img_file.parent
        imgs_in_dir = list(potential_dir.glob('*.jpg')) + list(potential_dir.glob('*.png'))
        if len(imgs_in_dir) > 5:  # 如果该目录有5张以上图片，认为是训练数据目录
            source_dir = potential_dir
            print(f'  ✅ 使用目录: {potential_dir} ({len(imgs_in_dir)} 张图片)')
            break

if source_dir is None:
    print('
❌ 未找到包含训练图片的目录！')
    print('请确认图片已上传到 Google Drive，路径示例:')
    print('  My Drive/lora_training_data/*.jpg')
    raise FileNotFoundError('Training data not found in Google Drive')

# 复制数据到训练目录
print(f'
📁 从 Google Drive 复制数据: {source_dir}')
img_count = 0
txt_count = 0

for img_file in list(source_dir.glob('*.jpg')) + list(source_dir.glob('*.png')) + list(source_dir.glob('*.jpeg')):
    shutil.copy(img_file, train_data_dir / img_file.name)
    img_count += 1

for txt_file in source_dir.glob('*.txt'):
    shutil.copy(txt_file, train_data_dir / txt_file.name)
    txt_count += 1

print(f'   ✅ 复制完成: {img_count} 张图片, {txt_count} 个标签')

# 检查数据完整性
image_files = list(train_data_dir.glob('*.jpg')) + list(train_data_dir.glob('*.png'))
txt_files = list(train_data_dir.glob('*.txt'))

print(f'
📊 训练数据统计：')
print(f'   图片文件: {len(image_files)} 个')
print(f'   标签文件: {len(txt_files)} 个')

if len(image_files) == 0:
    raise ValueError('没有找到图片文件！')
elif len(txt_files) == 0:
    print('
⚠️ 警告: 没有标签文件，将使用空标签继续训练')
elif len(image_files) != len(txt_files):
    print(f'
⚠️ 警告: 图片和标签数量不匹配！')
else:
    print(f'
✅ 数据完整！')

print(f'
示例文件:')
for f in list(image_files)[:3]:
    print(f'   {f.name}')

print('
' + '=' * 60)
print('✅ 训练数据准备完成！')
print('=' * 60)

In [ ]:
# （可选）Step 5 替代方案：从 Google Drive 挂载数据
from google.colab import drive
from pathlib import Path
import shutil

print("=" * 60)
print("📂 从 Google Drive 挂载数据")
print("=" * 60)

# 挂载 Google Drive
print("\n📂 挂载 Google Drive...")
drive.mount('/content/drive')
print("   ✅ 挂载成功！")

# 设置源目录（需要你修改！）
source_dir = Path("/content/drive/MyDrive/lora_training_data")  # 修改为你 Google Drive 中的目录

if not source_dir.exists():
    print(f"\n⚠️ 警告: 源目录不存在: {source_dir}")
    print(f"   请在 Google Drive 中创建目录: MyDrive/lora_training_data/")
    print(f"   并将训练数据（图片+标签）放入该目录")
else:
    # 复制到训练目录
    train_data_dir = Path("/content/kohya_ss/train_data/100_chezhuhuan")
    
    print(f"\n📁 复制数据: {source_dir} -> {train_data_dir}")
    
    # 复制图片和标签
    for img_file in list(source_dir.glob("*.jpg")) + list(source_dir.glob("*.png")):
        shutil.copy(img_file, train_data_dir / img_file.name)
    
    for txt_file in source_dir.glob("*.txt"):
        shutil.copy(txt_file, train_data_dir / txt_file.name)
    
    # 统计
    image_files = list(train_data_dir.glob("*.jpg")) + list(train_data_dir.glob("*.png"))
    txt_files = list(train_data_dir.glob("*.txt"))
    
    print(f"\n✅ 复制完成：")
    print(f"   图片文件: {len(image_files)} 个")
    print(f"   标签文件: {len(txt_files)} 个")

print("\n" + "=" * 60)

In [ ]:
# Step 6: 创建训练配置文件（SDXL LoRA）
from pathlib import Path

print("=" * 60)
print("⚙️ 创建训练配置文件")
print("=" * 60)

config_content = """
[model]
pretrained_model_name_or_path = "/content/kohya_ss/models/sd_xl_base_1.0.safetensors"
train_data_dir = "/content/kohya_ss/train_data"
output_dir = "/content/kohya_ss/output"
logging_dir = "/content/kohya_ss/logs"

[training]
train_batch_size = 2
gradient_accumulation_steps = 1
network_dim = 32
network_alpha = 16
max_train_epochs = 20
learning_rate = 1e-4
mixed_precision = "fp16"
save_every_n_epochs = 5
save_last_n_epochs = 3
seed = 42

[advanced]
resolution = "512,512"
min_bucket_reso = 256
max_bucket_reso = 1024
bucket_reso_steps = 64
flip_aug = true
caption_dropout_rate = 0.1
cache_latents = true
cache_text_encoder_outputs = true
use_8bit_adam = true
xformers = true
lowram = false
"""

# 保存配置文件
config_dir = Path("/content/kohya_ss/config")
config_file = config_dir / "chezhuhuan_sdxl.toml"

with open(config_file, "w") as f:
    f.write(config_content)

print(f"\n✅ 配置文件已创建: {config_file}")
print(f"\n配置内容:")
print("-" * 60)
print(config_content)
print("-" * 60)

print("\n" + "=" * 60)
print("✅ 配置完成！")
print("=" * 60)

In [ ]:
# Step 7: 测试训练环境（不实际训练，只检查配置）
from pathlib import Path
import subprocess

print("=" * 60)
print("🧪 测试训练环境")
print("=" * 60)

# 检查文件
print("\n🔍 检查文件...")
model_path = Path("/content/kohya_ss/models/sd_xl_base_1.0.safetensors")
train_script = Path("/content/kohya_ss/train_network.py")
config_file = Path("/content/kohya_ss/config/chezhuhuan_sdxl.toml")
train_data_dir = Path("/content/kohya_ss/train_data/100_chezhuhuan")

checks = [
    (model_path, "SDXL 模型"),
    (train_script, "训练脚本"),
    (config_file, "配置文件"),
]

all_passed = True
for path, name in checks:
    if path.exists():
        print(f"   ✅ {name}: {path}")
    else:
        print(f"   ❌ {name}: 不存在！")
        all_passed = False

# 检查训练数据
print(f"\n📊 训练数据:")
if train_data_dir.exists():
    image_files = list(train_data_dir.glob("*.jpg")) + list(train_data_dir.glob("*.png"))
    txt_files = list(train_data_dir.glob("*.txt"))
    print(f"   图片: {len(image_files)} 个")
    print(f"   标签: {len(txt_files)} 个")
    
    if len(image_files) > 0:
        print(f"   示例图片: {image_files[0].name}")
        print(f"   示例标签: {txt_files[0].name if txt_files else '无'}")
else:
    print(f"   ⚠️ 训练数据目录不存在: {train_data_dir}")
    all_passed = False

# 检查 GPU 显存
print(f"\n🎮 GPU 显存:")
import torch
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"   总显存: {gpu_memory:.2f} GB")
    
    if gpu_memory < 8:
        print(f"   ⚠️ 显存不足 8GB，建议修改配置:")
        print(f"      train_batch_size = 1")
        print(f"      gradient_accumulation_steps = 2")
    else:
        print(f"   ✅ 显存充足（SDXL LoRA 需要 8GB+）")

print("\n" + "=" * 60)
if all_passed:
    print("✅ 环境测试通过！可以开始训练！")
else:
    print("❌ 环境测试失败！请检查上面的错误。")
print("=" * 60)

In [ ]:
# Step 8: 开始训练（完整训练）
from pathlib import Path
import subprocess
import os

print("=" * 60)
print("🚀 开始 SDXL LoRA 训练")
print("=" * 60)

# 检查环境
config_file = Path("/content/kohya_ss/config/chezhuhuan_sdxl.toml")
train_script = Path("/content/kohya_ss/train_network.py")

if not config_file.exists():
    print(f"❌ 配置文件不存在: {config_file}")
    raise FileNotFoundError("Config file not found")

if not train_script.exists():
    print(f"❌ 训练脚本不存在: {train_script}")
    raise FileNotFoundError("Train script not found")

# 切换到 kohya_ss 目录
os.chdir("/content/kohya_ss")
print(f"
📂 当前目录: {os.getcwd()}")

# 构建训练命令
cmd = [
    "accelerate", "launch",
    "--num_cpu_threads_per_process", "2",
    "train_network.py",
    "--config_file", str(config_file)
]

print(f"
🚀 开始训练...")
print(f"   命令: {" ".join(cmd)}")
print(f"
⏳ 训练时间预估: 20 epochs ~ 1-2 小时")
print(f"   你可以关闭浏览器，训练会在后台继续")
print(f"   每 5 个 epoch 会自动保存一次模型
")
print("-" * 60)

# 启动训练
try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    
    # 实时输出训练日志
    for line in process.stdout:
        print(line, end="")
    
    process.wait()
    
    if process.returncode == 0:
        print("
" + "=" * 60)
        print("✅ 训练完成！")
        print("=" * 60)
    else:
        print(f"
❌ 训练失败！返回码: {process.returncode}")
        
except Exception as e:
    print(f"
❌ 训练出错: {e}")
    raise

print("
📌 下一步：下载训练好的 LoRA 模型")
print("   运行下一个单元格下载模型")


In [ ]:
# Step 9: 下载训练好的 LoRA 模型
from pathlib import Path
from google.colab import files

print("=" * 60)
print("📥 下载 LoRA 模型")
print("=" * 60)
    
# 查找训练好的模型
output_dir = Path("/content/kohya_ss/output")
lora_files = list(output_dir.glob("*.safetensors"))

if not lora_files:
    print("
⚠️ 没有找到训练好的 LoRA 模型！")
    print(f"   请检查输出目录: {output_dir}")
else:
    print(f"
✅ 找到 {len(lora_files)} 个 LoRA 模型:
")
    
    for i, lora_file in enumerate(lora_files):
        file_size = lora_file.stat().st_size / 1024**2
        print(f"   [{i+1}] {lora_file.name} ({file_size:.2f} MB)")
    
    print(f"
📥 开始下载...")
    
    # 下载所有模型
    for lora_file in lora_files:
        print(f"
下载: {lora_file.name}")
        files.download(str(lora_file))
    
    print(f"
✅ 所有模型已下载！")

print("
" + "=" * 60)
print("✅ 完成！")
print("=" * 60)
print("
💡 使用 LoRA 模型：")
print("   1. 在 Stable Diffusion WebUI 中加载 LoRA")
print("   2. 触发词: 车柱完")
print("   3. 调整 LoRA 权重（建议 0.6-0.8）")


## 📊 训练监控

训练过程中，你可以：

1. **查看 TensorBoard 日志**（可选）
   ```python
   %load_ext tensorboard
   %tensorboard --logdir /content/kohya_ss/logs
   ```

2. **检查输出目录**
   ```bash
   !ls -lh /content/kohya_ss/output/
   ```

3. **下载中间模型**
   - 每 5 个 epoch 会自动保存一次
   - 最后 3 个 epoch 的模型会被保留

---

## ⚠️ 常见问题

### 1. CUDA Out of Memory
**解决方法：**
- 修改配置文件：`train_batch_size = 1`
- 添加：`gradient_accumulation_steps = 2`
- 降低：`network_dim = 16`

### 2. 训练速度慢
**解决方法：**
- 确保安装了 xformers：`!pip install xformers`
- 在配置文件中启用：`xformers = true`

### 3. Colab 超时
**解决方法：**
- 免费版：12 小时限制，需要重新运行
- Pro 版：24 小时限制
- 建议：每 5 个 epoch 保存一次，断点续训

### 4. 数据上传失败
**解决方法：**
- 使用 Google Drive 挂载（更稳定）
- 或分批次上传数据

---

## 🎯 下一步

训练完成后：

1. **测试 LoRA 模型**
   - 在 Stable Diffusion WebUI 中加载
   - 使用触发词：`车柱完`
   - 生成测试图片

2. **调整参数（如需要）**
   - 如果过拟合：降低 `network_dim` 或增加 `caption_dropout_rate`
   - 如果欠拟合：增加 `max_train_epochs` 或降低 `learning_rate`

3. **分享你的模型！** 🎉
